# Round 2 · Graph ⚔️ tier B (30 min)

Rules see one account at a time. The campaigns still hidden touch many accounts with the *same* things: a browser profile (`device_id`), a canvas fingerprint (`fp_hash`), a provider identity (`provider_sub_hash`), a support agent. The log is a bipartite graph of accounts and identifiers; the attacker is a component in it.

Two moves: **seed expansion** from your Round 1 confirmed set, then **young, thin components**. And the legitimate hubs you must *not* flag: households on one desktop, 600 phones behind six carrier addresses, always-on VPNs, the QA team's shared browser, a person with two accounts on one Google identity, and a 250-account fake-account farm that is abuse but not takeover.

Cumulative: `suspects_r2.csv` includes your Round 1 accounts.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../tools"))
from atolab import *
import matplotlib.pyplot as plt
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

import networkx as nx
df = dedupe(load_events()); df = first_seen(df)
prev = pd.read_csv("../suspects_r1.csv")           # or the reference: python tools/checkpoint.py --round 1 --catch-up
DAY25 = pd.Timestamp("2026-10-26", tz="UTC")

## 2.1 First, the flood

Build the graph on `ja4` and `ip` and look at the biggest components. This is what a naive version finds, and why.

In [ ]:
G = build_bipartite(df, keys=("ja4", "ip"), idf=False, min_accounts=2)
P = project_accounts(G)
component_report(P, top=8)

## 2.2 Seed expansion

Every identifier that appeared on a Round 1 account **at or after** its first flagged event, and not before, is attacker infrastructure. Who else has it?

Two guards: only identifiers from events the *user* performed (the store's own policy engine touched the stuffing victims afterwards, from a service device shared by hundreds of accounts), and an identifier seen on more than a dozen accounts is infrastructure of some kind, not evidence.

In [ ]:
ev_ts = df.set_index("event_id").ts
user_rows = df[df.actor == "user"]
spread = {k: user_rows.groupby(k, observed=True).account_id.nunique() for k in ("device_id", "fp_hash")}
bad = set()
for r in prev.dropna(subset=["first_event_id"]).itertuples():
    t0 = ev_ts.get(r.first_event_id)
    a = user_rows[user_rows.account_id == r.account_id]
    for k in ("device_id", "fp_hash"):
        before = set(a.loc[a.ts < t0, k].dropna()); after = set(a.loc[a.ts >= t0, k].dropna())
        bad |= {(k, v) for v in after - before if spread[k].get(v, 0) <= 12}
exp = pd.concat([df[df[k] == v][["account_id", "event_id", "ts"]].assign(via=k) for k, v in bad]).sort_values("ts").drop_duplicates("account_id")
print(len(bad), "attacker identifiers ->", exp.account_id.nunique(), "accounts (", (~exp.account_id.isin(prev.account_id)).sum(), "new )")

## 2.3 Young, thin components

Build the graph on `device_id`, `fp_hash`, `d_provider_sub_hash`. Keep an identifier only if it links 2–12 accounts, every edge was first seen on or after day 25, and edges are thin (few events). Project, take components. Then look at each component's accounts with `timeline()` before you believe it.

Questions the code below leaves to you: what do a household's edges look like (age, thickness)? A duplicate-account owner's? The farm's? What is the one thing a *farm* account lacks that a *victim* has?

In [ ]:
G = build_bipartite(df, keys=("device_id", "fp_hash", "d_provider_sub_hash"), idf=False, min_accounts=2, max_accounts=12)
def old_or_thick(node, since=DAY25, max_events=8):
    e = [G[node][a] for a in G.neighbors(node)]
    return any(x["first_seen"] < since for x in e) or sorted(x["n_events"] for x in e)[len(e) // 2] > max_events
keep = [n for n, d in G.nodes(data=True) if d.get("kind") != "account" and not old_or_thick(n)]
# TODO: drop identifiers whose accounts ALSO share something old and thick (one person, two accounts)
H = G.subgraph([n for n, d in G.nodes(data=True) if d.get("kind") == "account"] + keep).copy()
H.remove_nodes_from([n for n in list(H.nodes) if H.degree(n) == 0])
comps = component_report(project_accounts(H), top=None)
comps.head(15)

In [ ]:
# TODO: separate abuse from takeover. An account whose first event in the log is account_created had no owner to take over.
first_type = df.sort_values("ts").groupby("account_id", observed=True).event_type.first()
comp_accounts = [a for accs in comps.accounts for a in accs]
takeover = [a for a in comp_accounts if first_type.get(a) != "account_created"]
print(len(comp_accounts), "accounts in components;", len(comp_accounts) - len(takeover), "look like a fake-account cluster")

## 2.4 Submit (cumulative) and score

In [ ]:
out = pd.concat([prev.assign(score=3.0), exp.assign(score=2.0)[["account_id", "first_event_id", "score"]] if "first_event_id" in exp else exp.rename(columns={"event_id": "first_event_id"}).assign(score=2.0)[["account_id", "first_event_id", "score"]],
                 pd.DataFrame({"account_id": takeover, "first_event_id": None, "score": 1.0})]).drop_duplicates("account_id")
save_suspects(out[["account_id"]], "../suspects_r2.csv", scores=out.score, first_event=out.first_event_id)
score_now("../suspects_r2.csv", "--prev", "../suspects_r1.csv")

## 2.5 Discuss

- Which weighting (rarity, age, thickness) did the most work? What would the attacker change next?
- One tier is still at zero. What do its victims share with each other? (Nothing. That is Round 3.)

Catch-up: `python tools/checkpoint.py --round 2 --catch-up`.